# Model Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

from io_utils import *
from clean import *
from models import *

import matplotlib.pyplot as plt

## Load and Clean Data

In [2]:
df_cleaned = load_raw_data()
df_cleaned = clean_data(df_cleaned)
df_minor = df_cleaned[df_cleaned["gcs_total"].isin([14, 15])].copy()

## Basic Information

In [3]:
print("Shape:", df_minor.shape)

Shape: (42430, 75)


## PECRARN


In [4]:
y_col = "citbi"

fold_metrics, avg_metrics = cv_rule_model(
    df_minor,
    y_col=y_col,
    predict_fn=pecarn_binary_predict,
    n_splits=10,
    seed=0,
)

save_table(fold_metrics, "pecarn_cv_results.csv")
save_table(avg_metrics, "pecarn_cv_summary.csv")
avg_metrics

,n,tp,tn,fp,fn,sensitivity,specificity,ppv,npv,accuracy,fpr,fnr
0,4241.2,36.6,2386.8,1816.8,1.0,0.973291,0.567798,0.019739,0.999581,0.571395,0.432202,0.026709


## Logistic Regression

In [5]:
missing_rate = df_minor.isna().mean().sort_values(ascending=False)
threshold = 0.3
cols_to_keep = missing_rate[missing_rate <= threshold].index
df_reduced = df_minor[cols_to_keep].copy()
df_model = df_reduced.dropna()

In [6]:
y_col = "citbi"
x_cols = [col for col in df_model.columns if col != y_col]

fold_metrics, avg_metrics = cv_trainable_model(
    df_model,
    y_col=y_col,
    x_cols=x_cols,
    fit_fn=fit_logistic,
    predict_fn=predict_logistic,
    n_splits=10,
    seed=0,
)

save_table(fold_metrics, "logistic_cv_results.csv")
save_table(avg_metrics, "logistic_cv_summary.csv")
avg_metrics

,n,tp,tn,fp,fn,sensitivity,specificity,ppv,npv,accuracy,fpr,fnr
0,2666.3,15.3,1250.9,1400.0,0.1,0.992308,0.471857,0.010945,0.999945,0.474894,0.528143,0.007692


## Hist Gradient Boosting Classifier

In [7]:
df_minor = df_minor.dropna(subset=[y_col])
y_col = "citbi"
x_cols = [c for c in df_minor.columns if c != y_col]


fold_metrics_hgb, avg_metrics_hgb = cv_trainable_model(
    df_minor,
    y_col=y_col,
    x_cols=x_cols,
    fit_fn=fit_hgb,
    predict_fn=lambda m, X: predict_hgb(m, X, threshold=0.01),
    n_splits=10,
    seed=0,
)
save_table(fold_metrics_hgb, "hgb_cv_results.csv")
save_table(avg_metrics_hgb, "hgb_cv_summary.csv")
avg_metrics_hgb

,n,tp,tn,fp,fn,sensitivity,specificity,ppv,npv,accuracy,fpr,fnr
0,4241.2,37.4,3875.6,328.0,0.2,0.995222,0.921971,0.102247,0.999948,0.922616,0.078029,0.004778
